<a href="https://colab.research.google.com/github/sarwathasan72-svg/Masai_IIT_AIML/blob/main/classification_evaluation_confusion_matrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification Evaluation: The Confusion Matrix
### Hands-on Notebook — Error Analysis, False Positives & False Negatives

**Learning Objectives**
1. Utilize the **Confusion Matrix** for error analysis
2. Identify **false positives and false negatives**

**Subtopics:** Confusion Matrix · TP / TN / FP / FN · Error Analysis

---
**The situation:** a teammate reports "90% accuracy" on a Pass/Fail model and calls it a win. Before agreeing, let's see what that number is hiding.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

plt.rcParams['figure.figsize'] = (5.5, 5)
np.random.seed(42)


## 1. Why accuracy alone can mislead

9 learners actually Pass, 1 actually Fails. A lazy model predicts "Pass" for everyone, never looking at the data.


In [ ]:
actual_lazy = np.array([1]*9 + [0])       # 1 = Pass, 0 = Fail
predicted_lazy = np.array([1]*10)          # always predicts Pass

acc_lazy = accuracy_score(actual_lazy, predicted_lazy)
print(f"Accuracy of the 'always predict Pass' model: {acc_lazy:.2f}")


90% accuracy — and the model is completely blind to the one Fail case. Let's build the tool that exposes this: the confusion matrix.


## 2. The confusion matrix — from scratch

For binary Pass(1)/Fail(0) labels, we can count the four outcomes directly.


In [ ]:
def manual_confusion_matrix(actual, predicted, positive_label=1):
    actual = np.array(actual)
    predicted = np.array(predicted)

    tp = np.sum((actual == positive_label) & (predicted == positive_label))
    tn = np.sum((actual != positive_label) & (predicted != positive_label))
    fp = np.sum((actual != positive_label) & (predicted == positive_label))
    fn = np.sum((actual == positive_label) & (predicted != positive_label))

    return {"TP": tp, "TN": tn, "FP": fp, "FN": fn}

manual_confusion_matrix(actual_lazy, predicted_lazy)


All 9 Pass predictions are correct (TP=9), but the single Fail case is misclassified as Pass — that's a **False Positive** (predicted Pass, actually Fail). Accuracy never showed us this; the confusion matrix does immediately.


## 3. Worked example — 10 learners

The same worked example from the slides.


In [ ]:
demo = pd.DataFrame({
    "learner": range(1, 11),
    "actual":    [1, 1, 1, 1, 1, 0, 0, 0, 0, 1],   # 1=Pass, 0=Fail
    "predicted": [1, 1, 1, 1, 0, 0, 0, 0, 1, 1],
})
demo


In [ ]:
result = manual_confusion_matrix(demo["actual"], demo["predicted"])
print(result)

acc = accuracy_score(demo["actual"], demo["predicted"])
print(f"Accuracy = (TP+TN)/total = {acc:.2f}")


**Checkpoint:** this should match the slide — TP=5, TN=3, FP=1, FN=1, accuracy=0.80.


## 4. Visualizing with scikit-learn

`ConfusionMatrixDisplay` renders the same grid from the slides, with counts filled in automatically.


In [ ]:
cm = confusion_matrix(demo["actual"], demo["predicted"], labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fail", "Pass"])
disp.plot(cmap="YlOrRd", colorbar=False)
plt.title("Confusion Matrix — Pass/Fail predictions")
plt.show()


## 5. Finding the specific errors

The confusion matrix tells you *how many* mistakes of each kind happened. To actually act on them, pull out *which* rows they were.


In [ ]:
false_positives = demo[(demo["actual"] == 0) & (demo["predicted"] == 1)]
false_negatives = demo[(demo["actual"] == 1) & (demo["predicted"] == 0)]

print("False Positives (predicted Pass, actually Fail):")
print(false_positives)

print("\nFalse Negatives (predicted Fail, actually Pass):")
print(false_negatives)


This is the real value of the matrix for error analysis — you don't just know the model made a mistake, you know exactly which learner and which type of mistake, so you can go look at *why*.


## 6. Cost asymmetry — why FP and FN aren't equal

Let's simulate two different domains with the same confusion matrix shape, to make the point that the "same" error count means different things depending on context.


In [ ]:
# Spam filter: positive class = "spam"
spam_actual =    np.array([1,1,1,0,0,0,0,0,0,0])   # 1 = spam, 0 = real email
spam_predicted = np.array([1,1,0,1,0,0,0,0,0,0])   # one FN (missed spam), one FP (real email flagged)

spam_cm = manual_confusion_matrix(spam_actual, spam_predicted)
print("Spam filter confusion counts:", spam_cm)
print("-> the FP here means a real email got buried in spam: often the costlier mistake.")

# Medical screening: positive class = "disease present"
med_actual =    np.array([1,1,1,0,0,0,0,0,0,0])
med_predicted = np.array([0,1,1,1,0,0,0,0,0,0])    # one FN (missed disease), one FP (false alarm)

med_cm = manual_confusion_matrix(med_actual, med_predicted)
print("\nMedical screening confusion counts:", med_cm)
print("-> the FN here means a real disease case was missed: often the costlier mistake.")


## 7. Exercises

Work through these before checking the solutions.


### Exercise 1 — Build a confusion matrix by hand
Given these actual/predicted labels, compute TP, TN, FP, FN by hand, then verify with `manual_confusion_matrix()`.

```
actual:    [1, 0, 1, 1, 0, 1, 0, 0]
predicted: [1, 0, 0, 1, 1, 1, 0, 0]
```


In [ ]:
# TODO: compute TP, TN, FP, FN manually, then check with manual_confusion_matrix()
ex1_actual = [1, 0, 1, 1, 0, 1, 0, 0]
ex1_predicted = [1, 0, 0, 1, 1, 1, 0, 0]


<details><summary>Show solution</summary>

```python
print(manual_confusion_matrix(ex1_actual, ex1_predicted))
# TP=3, TN=3, FP=1, FN=1
```
</details>


### Exercise 2 — Identify the costlier error
For a fraud-detection model (positive class = "transaction is fraudulent"), which error would you expect to be more costly to a bank — a False Positive or a False Negative? Write your reasoning in a markdown cell.


<details><summary>Show solution</summary>

There's no single universally "correct" answer, but a common line of reasoning: a **False Negative** (real fraud missed) directly costs the bank money and trust. A **False Positive** (a legitimate transaction blocked) costs customer convenience and support overhead, but is usually cheaper to recover from. Many fraud systems are tuned to accept more FPs in order to reduce FNs — the confusion matrix is what lets you see and defend that trade-off.
</details>


### Exercise 3 — Reproduce the "accuracy lies" example
Using `manual_confusion_matrix()`, confirm that the "always predict Pass" model from Section 1 has an FN count of 0 and an FP count of 1. What would the confusion matrix look like if, instead, the one actual Fail case had been correctly caught, at the cost of one actual Pass being wrongly flagged as Fail?


In [ ]:
# TODO: construct that alternative prediction array and compute its confusion matrix



<details><summary>Show solution</summary>

```python
alt_actual =    np.array([1]*9 + [0])
# real Fail (last case) now correctly caught, but it costs two actual Passes wrongly flagged as Fail
alt_predicted = np.array([0, 0] + [1]*7 + [0])
print(manual_confusion_matrix(alt_actual, alt_predicted))
print(accuracy_score(alt_actual, alt_predicted))
```
Catching that one real Fail case isn't free: to do it here the model also had to sacrifice two correct Pass predictions, and accuracy **drops** to 80%. This is the trade-off the confusion matrix makes visible — accuracy alone would tell you this "improved" model is actually worse.
</details>


## 8. Recap

- **Accuracy can hide serious problems** — a model that ignores the minority class entirely can still score high.
- The **confusion matrix** breaks every prediction into TP / TN / FP / FN, so successes and failures are visible separately.
- **False Positives** and **False Negatives** are different mistakes with different real-world costs — which one matters more always depends on the domain.
- Error analysis means reading the off-diagonal cells, deciding which error is costlier here, and acting on it (adjusting the threshold, gathering more data, or changing the model).

**Next up:** precision, recall, and F1 — metrics built directly on top of the confusion matrix.
